In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

# ============================================================
# ROADFLOOD-VLM
# Notebook 03: Scene Quality Assessment and Selection
# ============================================================

current_dir = Path.cwd()

if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

SEN1_DIR = RAW_DIR / "sen1floods11"
SPLITS_DIR = SEN1_DIR / "splits"
HANDLABELED_DIR = SEN1_DIR / "hand_labeled"

S1_DIR = HANDLABELED_DIR / "S1Hand"
S2_DIR = HANDLABELED_DIR / "S2Hand"
LABEL_DIR = HANDLABELED_DIR / "LabelHand"
JRC_DIR = HANDLABELED_DIR / "JRCWaterHand"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
MAP_DIR = OUTPUT_DIR / "maps"

for directory in [
    LABEL_DIR,
    JRC_DIR,
    TABLE_DIR,
    FIGURE_DIR,
    MAP_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

corrected_split_path = (
    TABLE_DIR
    / "sen1floods11_corrected_official_splits.csv"
)

if not corrected_split_path.exists():
    raise FileNotFoundError(
        "Corrected official split table not found. "
        "Run Notebook 01 first."
    )

official_splits_df = pd.read_csv(corrected_split_path)

print("=" * 72)
print("ROADFLOOD-VLM SCENE QUALITY ASSESSMENT")
print("=" * 72)

print(f"\nProject root       : {PROJECT_ROOT}")
print(f"Official scenes    : {len(official_splits_df):,}")
print(
    f"Unique scene IDs   : "
    f"{official_splits_df['scene_id'].nunique():,}"
)
print(f"Label directory    : {LABEL_DIR}")
print(f"JRC directory      : {JRC_DIR}")
print(f"Python version     : {sys.version.split()[0]}")

print("\n" + "=" * 72)
print("NOTEBOOK 03 INITIALIZATION COMPLETE")
print("=" * 72)

ROADFLOOD-VLM SCENE QUALITY ASSESSMENT

Project root       : C:\Users\takyi\Desktop\Machine Learning Engineering\ResilientVLM
Official scenes    : 446
Unique scene IDs   : 446
Label directory    : C:\Users\takyi\Desktop\Machine Learning Engineering\ResilientVLM\data\raw\sen1floods11\hand_labeled\LabelHand
JRC directory      : C:\Users\takyi\Desktop\Machine Learning Engineering\ResilientVLM\data\raw\sen1floods11\hand_labeled\JRCWaterHand
Python version     : 3.14.4

NOTEBOOK 03 INITIALIZATION COMPLETE


In [2]:
# ============================================================
# CELL 2: BUILD LIGHTWEIGHT SCENE-QUALITY MANIFEST
# ============================================================

import requests

print("=" * 72)
print("SCENE-QUALITY DOWNLOAD MANIFEST")
print("=" * 72)

BUCKET_NAME = "sen1floods11"
PUBLIC_BASE_URL = f"https://storage.googleapis.com/{BUCKET_NAME}"

QUALITY_COMPONENTS = {
    "LabelHand": {
        "folder": "LabelHand",
        "suffix": "LabelHand",
        "local_root": LABEL_DIR,
    },
    "JRCWaterHand": {
        "folder": "JRCWaterHand",
        "suffix": "JRCWaterHand",
        "local_root": JRC_DIR,
    },
}

quality_manifest_records = []

for _, scene_row in official_splits_df.iterrows():

    scene_id = scene_row["scene_id"]

    for component, config in QUALITY_COMPONENTS.items():

        filename = f"{scene_id}_{config['suffix']}.tif"

        object_name = (
            "v1.1/data/flood_events/HandLabeled/"
            f"{config['folder']}/{filename}"
        )

        local_path = (
            config["local_root"]
            / filename
        )

        quality_manifest_records.append(
            {
                "scene_id": scene_id,
                "country_prefix": scene_row["country_prefix"],
                "split": scene_row["split"],
                "component": component,
                "filename": filename,
                "object_name": object_name,
                "public_url": (
                    f"{PUBLIC_BASE_URL}/{object_name}"
                ),
                "local_path": str(local_path),
            }
        )

quality_manifest_df = pd.DataFrame(
    quality_manifest_records
)

print("\nMANIFEST SUMMARY")
print("-" * 72)

print(
    f"Unique scenes       : "
    f"{quality_manifest_df['scene_id'].nunique():,}"
)
print(
    f"Components per scene: "
    f"{quality_manifest_df['component'].nunique():,}"
)
print(f"Total files         : {len(quality_manifest_df):,}")

print("\nFiles by component:")

print(
    quality_manifest_df["component"]
    .value_counts()
    .to_string()
)

quality_manifest_path = (
    TABLE_DIR
    / "roadflood_vlm_scene_quality_manifest.csv"
)

quality_manifest_df.to_csv(
    quality_manifest_path,
    index=False,
)

print(f"\nSaved to: {quality_manifest_path}")

print("\n" + "=" * 72)
print("SCENE-QUALITY MANIFEST COMPLETE")
print("=" * 72)

SCENE-QUALITY DOWNLOAD MANIFEST

MANIFEST SUMMARY
------------------------------------------------------------------------
Unique scenes       : 446
Components per scene: 2
Total files         : 892

Files by component:
component
LabelHand       446
JRCWaterHand    446

Saved to: C:\Users\takyi\Desktop\Machine Learning Engineering\ResilientVLM\outputs\tables\roadflood_vlm_scene_quality_manifest.csv

SCENE-QUALITY MANIFEST COMPLETE


In [3]:
# ============================================================
# CELL 3: DOWNLOAD LABELHAND AND JRCWATERHAND FILES
# ============================================================

from pathlib import Path
import requests
from tqdm.auto import tqdm
import pandas as pd

print("=" * 72)
print("SCENE-QUALITY RASTER DOWNLOAD")
print("=" * 72)


def download_small_raster(
    url,
    output_path,
    timeout=120,
):
    """
    Download one small raster, skipping valid existing files.
    """

    output_path = Path(output_path)
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if output_path.exists() and output_path.stat().st_size > 0:
        return {
            "status": "SKIPPED",
            "size_bytes": output_path.stat().st_size,
            "error": None,
        }

    temporary_path = output_path.with_suffix(
        output_path.suffix + ".part"
    )

    if temporary_path.exists():
        temporary_path.unlink()

    try:

        with requests.get(
            url,
            stream=True,
            timeout=timeout,
        ) as response:

            response.raise_for_status()

            with open(temporary_path, "wb") as file_handle:

                for chunk in response.iter_content(
                    chunk_size=1024 * 256
                ):
                    if chunk:
                        file_handle.write(chunk)

        temporary_path.replace(output_path)

        return {
            "status": "DOWNLOADED",
            "size_bytes": output_path.stat().st_size,
            "error": None,
        }

    except Exception as exc:

        if temporary_path.exists():
            temporary_path.unlink()

        return {
            "status": "FAILED",
            "size_bytes": 0,
            "error": str(exc),
        }


download_records = []

for _, row in tqdm(
    quality_manifest_df.iterrows(),
    total=len(quality_manifest_df),
    desc="Downloading scene-quality rasters",
):

    result = download_small_raster(
        row["public_url"],
        row["local_path"],
    )

    download_records.append(result)


quality_download_df = pd.concat(
    [
        quality_manifest_df.reset_index(drop=True),
        pd.DataFrame(download_records),
    ],
    axis=1,
)

print("\nDOWNLOAD SUMMARY")
print("-" * 72)

print(
    quality_download_df["status"]
    .value_counts()
    .to_string()
)

valid_files = (
    quality_download_df["size_bytes"] > 0
).sum()

failed_files = (
    quality_download_df["status"] == "FAILED"
).sum()

print(f"\nValid local files : {valid_files:,}")
print(f"Failed files      : {failed_files:,}")
print(
    f"Total local size  : "
    f"{quality_download_df['size_bytes'].sum() / (1024 ** 2):,.2f} MB"
)

download_output = (
    TABLE_DIR
    / "roadflood_vlm_scene_quality_downloads.csv"
)

quality_download_df.to_csv(
    download_output,
    index=False,
)

print(f"\nSaved to: {download_output}")

if failed_files > 0:

    print("\nFAILED DOWNLOADS")
    print("-" * 72)

    print(
        quality_download_df.loc[
            quality_download_df["status"] == "FAILED",
            [
                "scene_id",
                "component",
                "error",
            ],
        ].to_string(index=False)
    )

print("\n" + "=" * 72)

if failed_files == 0:
    print("SCENE-QUALITY DOWNLOAD COMPLETE")
else:
    print("SCENE-QUALITY DOWNLOAD REQUIRES REVIEW")

print("=" * 72)

C:\Users\takyi\Desktop\Machine Learning Engineering\ResilientVLM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SCENE-QUALITY RASTER DOWNLOAD



DOWNLOAD SUMMARY
------------------------------------------------------------------------
status
DOWNLOADED    892

Valid local files : 892
Failed files      : 0
Total local size  : 2.93 MB

Saved to: C:\Users\takyi\Desktop\Machine Learning Engineering\ResilientVLM\outputs\tables\roadflood_vlm_scene_quality_downloads.csv

SCENE-QUALITY DOWNLOAD COMPLETE


In [5]:
# ============================================================
# CELL 4: COMPUTE SCENE-QUALITY STATISTICS
# ============================================================

import rasterio
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 72)
print("SCENE-QUALITY METRIC COMPUTATION")
print("=" * 72)

scene_quality_records = []

for _, scene_row in official_splits_df.iterrows():

    scene_id = scene_row["scene_id"]

    label_path = (
        LABEL_DIR
        / f"{scene_id}_LabelHand.tif"
    )

    jrc_path = (
        JRC_DIR
        / f"{scene_id}_JRCWaterHand.tif"
    )

    if not label_path.exists():
        continue

    with rasterio.open(label_path) as src:

        label = np.asarray(src.read(1)).copy()

        bounds = src.bounds
        crs = str(src.crs)

    total_pixels = label.size

    ignore_pixels = int(
        np.count_nonzero(label == -1)
    )

    nonflood_pixels = int(
        np.count_nonzero(label == 0)
    )

    flood_pixels = int(
        np.count_nonzero(label == 1)
    )

    valid_pixels = (
        nonflood_pixels
        + flood_pixels
    )

    ignore_pct = (
        100 * ignore_pixels / total_pixels
    )

    valid_pct = (
        100 * valid_pixels / total_pixels
    )

    flood_pct_total = (
        100 * flood_pixels / total_pixels
    )

    flood_pct_valid = (
        100 * flood_pixels / valid_pixels
        if valid_pixels > 0
        else 0.0
    )

    nonflood_pct_valid = (
        100 * nonflood_pixels / valid_pixels
        if valid_pixels > 0
        else 0.0
    )

    permanent_water_pixels = 0
    permanent_water_pct = 0.0

    if jrc_path.exists():

        with rasterio.open(jrc_path) as src:
            jrc = np.asarray(src.read(1)).copy()

        permanent_water_pixels = int(
            np.count_nonzero(jrc == 1)
        )

        permanent_water_pct = (
            100
            * permanent_water_pixels
            / total_pixels
        )

    scene_quality_records.append(
        {
            "scene_id": scene_id,
            "country_prefix": scene_row["country_prefix"],
            "split": scene_row["split"],
            "crs": crs,
            "left": bounds.left,
            "bottom": bounds.bottom,
            "right": bounds.right,
            "top": bounds.top,
            "total_pixels": total_pixels,
            "ignore_pixels": ignore_pixels,
            "valid_pixels": valid_pixels,
            "nonflood_pixels": nonflood_pixels,
            "flood_pixels": flood_pixels,
            "permanent_water_pixels": permanent_water_pixels,
            "ignore_pct": ignore_pct,
            "valid_pct": valid_pct,
            "flood_pct_total": flood_pct_total,
            "flood_pct_valid": flood_pct_valid,
            "nonflood_pct_valid": nonflood_pct_valid,
            "permanent_water_pct": permanent_water_pct,
        }
    )

scene_quality_df = pd.DataFrame(
    scene_quality_records
)

print("\nSCENE STATISTICS SUMMARY")
print("-" * 72)

print(f"Scenes processed : {len(scene_quality_df):,}")

summary_columns = [
    "ignore_pct",
    "valid_pct",
    "flood_pct_total",
    "flood_pct_valid",
    "permanent_water_pct",
]

print(
    scene_quality_df[summary_columns]
    .describe()
    .round(3)
    .to_string()
)

print("\nScenes containing flood pixels:")

print(
    (
        scene_quality_df["flood_pixels"] > 0
    ).value_counts().to_string()
)

quality_stats_output = (
    TABLE_DIR
    / "roadflood_vlm_scene_quality_statistics.csv"
)

scene_quality_df.to_csv(
    quality_stats_output,
    index=False,
)

print(f"\nSaved to: {quality_stats_output}")

print("\n" + "=" * 72)
print("SCENE-QUALITY METRICS COMPLETE")
print("=" * 72)

SCENE-QUALITY METRIC COMPUTATION


C:\Users\takyi\AppData\Local\Temp\ipykernel_8276\3650998500.py:35: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  label = np.asarray(src.read(1)).copy()
C:\Users\takyi\AppData\Local\Temp\ipykernel_8276\3650998500.py:89: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  jrc = np.asarray(src.read(1)).copy()



SCENE STATISTICS SUMMARY
------------------------------------------------------------------------
Scenes processed : 446
       ignore_pct  valid_pct  flood_pct_total  flood_pct_valid  permanent_water_pct
count     446.000    446.000          446.000          446.000              446.000
mean       13.628     86.372            9.157           10.749                3.065
std        22.237     22.237           16.431           19.073               10.201
min         0.000      0.000            0.000            0.000                0.000
25%         0.002     80.782            0.461            0.512                0.000
50%         0.949     99.051            2.188            2.504                0.003
75%        19.218     99.998            9.844           11.278                0.554
max       100.000    100.000           98.140          100.000               97.741

Scenes containing flood pixels:
flood_pixels
True     394
False     52

Saved to: C:\Users\takyi\Desktop\Machine Learning

In [6]:
# ============================================================
# CELL 5: ELIGIBILITY FILTERING AND SCENE RANKING
# ============================================================

print("=" * 72)
print("ROADFLOOD-VLM SCENE RANKING")
print("=" * 72)

# ------------------------------------------------------------
# 1. Eligibility criteria
# ------------------------------------------------------------

MIN_VALID_PCT = 80.0
MIN_FLOOD_VALID_PCT = 1.0
MAX_FLOOD_VALID_PCT = 60.0
MAX_PERMANENT_WATER_PCT = 20.0

scene_quality_df["eligible"] = (
    (scene_quality_df["valid_pct"] >= MIN_VALID_PCT)
    &
    (
        scene_quality_df["flood_pct_valid"]
        >= MIN_FLOOD_VALID_PCT
    )
    &
    (
        scene_quality_df["flood_pct_valid"]
        <= MAX_FLOOD_VALID_PCT
    )
    &
    (
        scene_quality_df["permanent_water_pct"]
        <= MAX_PERMANENT_WATER_PCT
    )
)

# ------------------------------------------------------------
# 2. Component scores
# ------------------------------------------------------------

# Valid data score: 0–100
scene_quality_df["valid_score"] = (
    scene_quality_df["valid_pct"]
    .clip(0, 100)
)

# Flood balance score:
# Peaks at approximately 20% flood among valid pixels.
TARGET_FLOOD_PCT = 20.0

scene_quality_df["flood_balance_score"] = (
    100
    -
    (
        abs(
            scene_quality_df["flood_pct_valid"]
            - TARGET_FLOOD_PCT
        )
        / TARGET_FLOOD_PCT
        * 100
    )
).clip(0, 100)

# Penalize permanent water contamination
scene_quality_df["water_separation_score"] = (
    100
    -
    (
        scene_quality_df["permanent_water_pct"]
        * 5
    )
).clip(0, 100)

# Reward scenes with both flood and non-flood evidence
scene_quality_df["class_balance_score"] = (
    100
    -
    abs(
        scene_quality_df["flood_pct_valid"]
        -
        scene_quality_df["nonflood_pct_valid"]
    )
).clip(0, 100)

# ------------------------------------------------------------
# 3. Preliminary suitability score
# ------------------------------------------------------------

scene_quality_df["preliminary_suitability_score"] = (
    0.35 * scene_quality_df["valid_score"]
    +
    0.35 * scene_quality_df["flood_balance_score"]
    +
    0.20 * scene_quality_df["water_separation_score"]
    +
    0.10 * scene_quality_df["class_balance_score"]
)

scene_quality_df.loc[
    ~scene_quality_df["eligible"],
    "preliminary_suitability_score",
] = 0.0

scene_quality_df["quality_rank"] = (
    scene_quality_df[
        "preliminary_suitability_score"
    ]
    .rank(
        method="dense",
        ascending=False,
    )
    .astype(int)
)

ranked_scene_df = (
    scene_quality_df
    .sort_values(
        [
            "eligible",
            "preliminary_suitability_score",
            "flood_pct_valid",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

print("\nELIGIBILITY SUMMARY")
print("-" * 72)

print(
    ranked_scene_df["eligible"]
    .value_counts()
    .to_string()
)

print("\nTOP 30 PRELIMINARY SCENES")
print("-" * 72)

display_columns = [
    "quality_rank",
    "scene_id",
    "country_prefix",
    "split",
    "valid_pct",
    "ignore_pct",
    "flood_pct_valid",
    "permanent_water_pct",
    "preliminary_suitability_score",
]

print(
    ranked_scene_df.loc[
        ranked_scene_df["eligible"],
        display_columns,
    ]
    .head(30)
    .round(3)
    .to_string(index=False)
)

ranking_output = (
    TABLE_DIR
    / "roadflood_vlm_preliminary_scene_ranking.csv"
)

ranked_scene_df.to_csv(
    ranking_output,
    index=False,
)

print(f"\nSaved to: {ranking_output}")

print("\n" + "=" * 72)
print("PRELIMINARY SCENE RANKING COMPLETE")
print("=" * 72)

ROADFLOOD-VLM SCENE RANKING

ELIGIBILITY SUMMARY
------------------------------------------------------------------------
eligible
False    238
True     208

TOP 30 PRELIMINARY SCENES
------------------------------------------------------------------------
 quality_rank        scene_id country_prefix      split  valid_pct  ignore_pct  flood_pct_valid  permanent_water_pct  preliminary_suitability_score
            1   Spain_8565131          Spain validation     99.977       0.023           19.977                0.380                         93.567
            2    India_804466          India      train     99.996       0.004           19.161                0.347                         92.016
            3   India_1050276          India validation     97.345       2.655           19.204                1.857                         89.661
            4 Nigeria_1095404        Nigeria validation     96.238       3.762           18.270                0.003                         89.306
   